# Notebook 02: Scenario Analysis and Stress Testing

**Project:** Brent Crude Oil Prices and Renewable Energy Investment  
**Date:** 2026-04-06  

This notebook runs the CES investment model across five calibrated scenarios
and produces investment trajectory charts and summary tables.

## Scenarios

| Scenario | 2025 ($/bbl) | 2030 ($/bbl) | Source |
|---|---|---|---|
| STEPS (baseline) | 75 | 65 | IEA WEO 2024 Stated Policies |
| APS (pledges) | 70 | 55 | IEA WEO 2024 Announced Pledges |
| NZE (net zero) | 65 | 44 | IEA WEO 2024 Net Zero 2050 |
| HIGH_SHOCK | 90 | 130 | IMF WP/23/160 (Boer et al. 2023) |
| LOW_SHOCK | 60 | 25 | Boer et al. 2023 demand-side policy |


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from ces_model.core.ces import CESModel
from ces_model.scenarios.paths import SCENARIOS, scenarios_to_dataframe
from ces_model.scenarios.stress_test import StressTest

import os
os.makedirs("figures", exist_ok=True)

# Calibrated model (alpha calibrated to 30% share at $80/bbl)
model = CESModel(sigma=1.8)
model.calibrate_alpha(observed_share=0.30, price_oil=80.0, price_renew=1.0)
print(f"Calibrated alpha: {model.alpha:.6f}")

## 1. Scenario Price Paths (2020-2030)

In [ ]:
sc_df = scenarios_to_dataframe()
print(sc_df.to_string(index=False))

In [ ]:
colors = {
    "STEPS": "steelblue",
    "APS": "mediumseagreen",
    "NZE": "forestgreen",
    "HIGH_SHOCK": "firebrick",
    "LOW_SHOCK": "darkorange",
}
styles = {
    "STEPS": "-",
    "APS": "--",
    "NZE": "-.",
    "HIGH_SHOCK": "-",
    "LOW_SHOCK": "--",
}

fig, ax = plt.subplots(figsize=(10, 5))
for name, sc in SCENARIOS.items():
    df = sc.to_dataframe()
    proj = df[df["year"] >= 2024]
    hist = df[df["year"] <= 2024]
    ax.plot(hist["year"], hist["price_usd_bbl"], color="dimgray",
            linewidth=1.5, zorder=5)
    ax.plot(proj["year"], proj["price_usd_bbl"],
            color=colors[name], linestyle=styles[name],
            linewidth=2, label=f"{name}: {sc.label}")

ax.axvline(2024.5, color="dimgray", linestyle=":", linewidth=0.8)
ax.text(2024.55, 5, "Projection >", fontsize=8, color="dimgray")
ax.set_xlabel("Year")
ax.set_ylabel("Brent Price (USD/bbl)")
ax.set_title("Five-Scenario Brent Oil Price Paths (2020-2030)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/02a_scenario_price_paths.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Investment Trajectories Under Each Scenario

In [ ]:
# Run stress test with calibrated model
st = StressTest(
    ces_model=model,
    base_invest_bn_usd=807.0,
    policy_multiplier=1.0,
    capex_decline_rate=0.20,
    projection_start=2025,
)
result = st.run()

# Show summary table
summary = result.summary()
print(summary.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, sr in result.scenario_results.items():
    df = sr.to_dataframe()
    ax.plot(df["year"], df["policy_adj_invest_bn_usd"],
            color=colors[name], linestyle=styles[name],
            linewidth=2, marker="o", markersize=5,
            label=f"{name}")

ax.axhline(807.0, color="dimgray", linestyle=":", linewidth=0.8,
           label="2024 base ($807 bn)")
ax.set_xlabel("Year")
ax.set_ylabel("Renewable Power Investment (USD bn)")
ax.set_title("CES Model: Renewable Investment Trajectories 2025-2030")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/02b_investment_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Renewable Share Trajectories

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, sr in result.scenario_results.items():
    df = sr.to_dataframe()
    ax.plot(df["year"], df["renew_share"],
            color=colors[name], linestyle=styles[name],
            linewidth=2, marker="s", markersize=5,
            label=f"{name}")

ax.set_xlabel("Year")
ax.set_ylabel("Renewable Share (CES model)")
ax.set_title("CES Model: Renewable Energy Share Trajectories 2025-2030")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/02c_share_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Fan Chart: Investment Range Across Scenarios

In [ ]:
wide_df = result.to_wide_dataframe()
years = wide_df["year"].values

invest_cols = [c for c in wide_df.columns if c != "year"]
invest_mat = wide_df[invest_cols].values

fig, ax = plt.subplots(figsize=(10, 5))

# Fill between min/max
ax.fill_between(
    years,
    invest_mat.min(axis=1),
    invest_mat.max(axis=1),
    alpha=0.15, color="steelblue", label="Min-Max range"
)

# STEPS baseline
ax.plot(years, wide_df["STEPS"], color="steelblue", linewidth=2.5,
        label="STEPS (baseline)", zorder=5)

# Stress scenarios
for name in ["NZE", "HIGH_SHOCK", "LOW_SHOCK"]:
    ax.plot(years, wide_df[name], color=colors[name], linewidth=1.5,
            linestyle="--", label=name)

ax.axhline(807.0, color="dimgray", linestyle=":", linewidth=0.8,
           label="2024 base ($807 bn)")
ax.set_xlabel("Year")
ax.set_ylabel("Renewable Power Investment (USD bn)")
ax.set_title("CES Stress Test: Investment Fan Chart 2025-2030")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/02d_investment_fan_chart.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary at 2030
print("\n2030 Terminal Investment Summary:")
for name in invest_cols:
    val = wide_df[wide_df["year"] == 2030.0][name].values[0]
    print(f"  {name:<12}: ${val:.1f} bn")

## 5. Policy Multiplier Sensitivity

Illustrate the effect of a policy uplift (e.g., IRA/REPowerEU) on the
investment trajectory under the STEPS baseline.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
policy_multipliers = [1.0, 1.2, 1.5, 2.0]
pm_colors = ["steelblue", "mediumseagreen", "darkorange", "firebrick"]

for pm, col in zip(policy_multipliers, pm_colors):
    st_pm = StressTest(
        ces_model=model,
        base_invest_bn_usd=807.0,
        policy_multiplier=pm,
        capex_decline_rate=0.20,
    )
    sr = st_pm.run_single("STEPS")
    df = sr.to_dataframe()
    ax.plot(df["year"], df["policy_adj_invest_bn_usd"],
            color=col, linewidth=2, marker="o",
            label=f"Policy multiplier x{pm}")

ax.axhline(807.0, color="dimgray", linestyle=":", linewidth=0.8)
ax.set_xlabel("Year")
ax.set_ylabel("Renewable Investment (USD bn)")
ax.set_title("STEPS Baseline: Effect of Policy Multiplier on Investment")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/02e_policy_multiplier.png", dpi=150, bbox_inches="tight")
plt.show()